# BankAssist Triage — MLflow experiments and model registry

Train two banking-intent classifiers, compare them in MLflow, register the selected pipeline, and promote a validated version to `@champion`.

## 1. Experiment setup

- **Data:** use the fixed `train.csv`, `validation.csv`, and `test.csv` in `../data/splits/`. The input is `text`; the target is `label`.
- **Models:** both are TF-IDF + `LogisticRegression(max_iter=1000, random_state=42)` pipelines. Model A uses unigrams (`ngram_range=(1, 1)`); Model B uses unigrams and bigrams (`ngram_range=(1, 2)`). Other settings use scikit-learn defaults.
- **Selection:** train on `train.csv` and choose the higher **validation macro F1**; report validation accuracy too. If tied, choose Model A. Use `test.csv` only for the selected model's final evaluation.
- **MLflow:** log one run per model with dataset information, parameters, validation metrics, a confusion matrix, prediction samples, and the fitted pipeline. Register the winner as `@candidate`, validate the loaded model, then assign `@champion` if the promotion checks pass.

## 2. Load and check the fixed splits

Load the saved splits and check their columns, labels, and text overlap. Run the notebook from the repository root or its `notebooks` directory.

In [ ]:
from pathlib import Path

import pandas as pd

project_dir = Path.cwd()
if not (project_dir / 'data' / 'splits').is_dir():
    project_dir = project_dir.parent
split_dir = project_dir / 'data' / 'splits'
assert split_dir.is_dir(), f'Cannot find data/splits from {Path.cwd()}'

train_df = pd.read_csv(split_dir / 'train.csv')
validation_df = pd.read_csv(split_dir / 'validation.csv')
test_df = pd.read_csv(split_dir / 'test.csv')

for split_name, frame in [('train', train_df), ('validation', validation_df), ('test', test_df)]:
    assert {'text', 'label'} <= set(frame.columns), f'{split_name}: missing required columns'
    assert frame[['text', 'label']].notna().all().all(), f'{split_name}: missing text or label'
    print(f"{split_name}: {len(frame)} rows, {frame['label'].nunique()} intents")

assert set(validation_df['label']) <= set(train_df['label'])
assert set(test_df['label']) <= set(train_df['label'])
assert not (set(train_df['text']) & set(validation_df['text']))
assert not (set(train_df['text']) & set(test_df['text']))
assert not (set(validation_df['text']) & set(test_df['text']))
print('Required columns, labels, and cross-split text checks passed.')

## 3. Connect to the MLflow tracking server

Start the service with `docker compose up -d mlflow`. This VS Code notebook connects at `http://localhost:5001`; an API container in the same Compose project can use `http://mlflow:5000`. Runs and artifacts persist in the Git-ignored `mlflow-data/` folder.

In [ ]:
import os

import mlflow

TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5001')
EXPERIMENT_NAME = 'BankAssist-Intent-Classification'
REGISTERED_MODEL_NAME = 'BankAssist-Intent-Classifier'

mlflow.set_tracking_uri(TRACKING_URI)
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
print('MLflow tracking URI:', mlflow.get_tracking_uri())
print('Experiment:', experiment.name, '(id:', experiment.experiment_id + ')')
print('Registered model name for later steps:', REGISTERED_MODEL_NAME)

## 4. Train and track Models A and B

Each model gets its own MLflow run. The run records the data, settings, validation results, confusion matrix, prediction samples, and fitted pipeline. The test set is not used in this comparison.

In [ ]:
import matplotlib.pyplot as plt
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
)
from sklearn.pipeline import Pipeline

X_train, y_train = train_df['text'], train_df['label']
X_validation, y_validation = validation_df['text'], validation_df['label']
class_names = sorted(y_train.unique())

model_configs = [
    {'name': 'Model A - unigrams', 'ngram_range': (1, 1)},
    {'name': 'Model B - unigrams and bigrams', 'ngram_range': (1, 2)},
]

# Creates the MLflow Dataset objects:
train_input = mlflow.data.from_pandas(
    train_df, source=str((split_dir / 'train.csv').resolve()),
    name='bankassist_train', targets='label'
)
validation_input = mlflow.data.from_pandas(
    validation_df, source=str((split_dir / 'validation.csv').resolve()),
    name='bankassist_validation', targets='label'
)
print('Ready to train', len(model_configs), 'models across', len(class_names), 'intents.')

Ready to train 2 models across 10 intents.


c:\Users\Jonny\Documents\Joaninha\EXEC MASTERS IN BUSINESS ANALYTICS AND AI\MLOps\bankassist-triage\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\Jonny\Documents\Joaninha\EXEC MASTERS IN BUSINESS ANALYTICS AND AI\MLOps\bankassist-triage\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


In [7]:
run_results = []

for config in model_configs:
    with mlflow.start_run(run_name=config['name']) as run:
        mlflow.log_input(train_input, context='training')
        mlflow.log_input(validation_input, context='validation')
        mlflow.log_params({
            'model': config['name'],
            'tfidf_ngram_range': str(config['ngram_range']),
            'classifier': 'LogisticRegression',
            'max_iter': 1000,
            'random_state': 42,
            'train_rows': len(train_df),
            'validation_rows': len(validation_df),
        })

        pipeline = Pipeline([
            ('tfidf', TfidfVectorizer(ngram_range=config['ngram_range'])),
            ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
        ])
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_validation)
        macro_f1 = f1_score(y_validation, predictions, average='macro')
        accuracy = accuracy_score(y_validation, predictions)
        mlflow.log_metrics({
            'validation_macro_f1': macro_f1,
            'validation_accuracy': accuracy,
        })

        fig, ax = plt.subplots(figsize=(12, 10))
        ConfusionMatrixDisplay.from_predictions(
            y_validation, predictions, labels=class_names,
            display_labels=class_names, xticks_rotation=90,
            cmap='Blues', colorbar=False, ax=ax
        )
        fig.tight_layout()
        mlflow.log_figure(fig, 'validation/confusion_matrix.png')
        plt.close(fig)
        mlflow.log_dict(
            classification_report(y_validation, predictions, output_dict=True),
            'validation/classification_report.json'
        )
        samples = pd.DataFrame({
            'text': X_validation.to_numpy(),
            'actual_intent': y_validation.to_numpy(),
            'predicted_intent': predictions,
            'confidence': pipeline.predict_proba(X_validation).max(axis=1),
        }).head(20)
        mlflow.log_text(samples.to_csv(index=False), 'validation/prediction_samples.csv')

        example_texts = X_train.iloc[:3].to_numpy()
        signature = infer_signature(example_texts, pipeline.predict(example_texts))
        model_info = mlflow.sklearn.log_model(
            pipeline, name='intent_classifier', signature=signature,
            input_example=example_texts
        )
        mlflow.set_tag('logged_model_id', model_info.model_id)
        run_results.append({
            'model': config['name'],
            'validation_macro_f1': macro_f1,
            'validation_accuracy': accuracy,
            'run_id': run.info.run_id,
            'model_id': model_info.model_id,
        })
        print(f"{config['name']}: macro F1={macro_f1:.4f}, accuracy={accuracy:.4f}")

Model A - unigrams: macro F1=0.9194, accuracy=0.9189
🏃 View run Model A - unigrams at: http://localhost:5001/#/experiments/1/runs/d8a6d09807e74267a643c038cb2ab659
🧪 View experiment at: http://localhost:5001/#/experiments/1
Model B - unigrams and bigrams: macro F1=0.9026, accuracy=0.9054
🏃 View run Model B - unigrams and bigrams at: http://localhost:5001/#/experiments/1/runs/cfec607c64b343bcb14c8b4d1a37f4eb
🧪 View experiment at: http://localhost:5001/#/experiments/1


scikit-learn: performs the training.

MLFlow: tracks the training.

Every time we run the cell above, a new run is registered in MLFlow, separately (previous ones are not overwritten).

In MLFlow:
- **Green checks:** show that the run completed.
- **Dataset Feature:** records dataset's identity, source, schema and fingerprint.
- **Models:** "intent classifier" - it is the logged model (not yet the registerd @candidate or @champion)

By clicking on one Model A row and one Model B row I can compare their Parameters and Metrics.

![Comparison of Model A and Model B in MLflow](../docs/images/1_model_runs.png)

## 5. Compare validation runs

Select by validation macro F1. The run and model IDs identify the exact logged pipeline for the registry step.

In [8]:
comparison = pd.DataFrame(run_results).sort_values(
    ['validation_macro_f1', 'model'], ascending=[False, True]
).reset_index(drop=True)
print(comparison.to_string(index=False))

winner = comparison.iloc[0]
print(f"Selected: {winner['model']} (run {winner['run_id']})")

                         model  validation_macro_f1  validation_accuracy                           run_id                           model_id
            Model A - unigrams             0.919369             0.918919 d8a6d09807e74267a643c038cb2ab659 m-717d260686c84caf86f4aa528d0e7bb7
Model B - unigrams and bigrams             0.902560             0.905405 cfec607c64b343bcb14c8b4d1a37f4eb m-eefb424487664312976580544a6809c2
Selected: Model A - unigrams (run d8a6d09807e74267a643c038cb2ab659)


![Comparison of Model A and Model B in MLflow](../docs/images/metrics_comparison_1.png)

![Comparison of Model A and Model B in MLflow](../docs/images/metrics_comparison_2.png)

## 6. Register the selected model

The selected run (in this case, Model A) already contains a logged pipeline. 

Registering it under `BankAssist-Intent-Classifier` creates a numbered model version linked to that run (MLflow will create a numbered version that will keep a link back to the run that produced it.). 

The `@candidate` alias marks the version to validate next (it means that this is the version that we want to check next.); it is not yet the model served by the API. 

Later, after the checks pass, we move the @champion label to the approved version. This is the version that our app will use.

**SUMMARY**

- The RUN produced the model
- The REGISTRY gave the model a version
- @candidate identifies the version awaiting checks
- @champion identifies the approved version

In [10]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
selected_run_id = str(winner['run_id'])
selected_model_id = str(winner['model_id'])
selected_model_uri = f'models:/{selected_model_id}'
selected_run = client.get_run(selected_run_id)
assert selected_run.info.status == 'FINISHED'
assert selected_run.data.tags['logged_model_id'] == selected_model_id

# Reuse the version if this exact logged model was registered on an earlier cell run.
versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
matches = [version for version in versions if version.source == selected_model_uri]
if matches:
    model_version = max(matches, key=lambda version: int(version.version))
    print('Reusing registered version', model_version.version)
else:
    model_version = mlflow.register_model(
        model_uri=selected_model_uri,
        name=REGISTERED_MODEL_NAME,
    )
    print('Created registered version', model_version.version)

client.set_model_version_tag(
    REGISTERED_MODEL_NAME, model_version.version,
    'validation_macro_f1', f"{winner['validation_macro_f1']:.4f}"
)
client.set_registered_model_alias(
    REGISTERED_MODEL_NAME, 'candidate', model_version.version
)
candidate = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, 'candidate')
print(f"@candidate -> {REGISTERED_MODEL_NAME} version {candidate.version}")
print('Source run:', candidate.run_id)
print('Candidate URI:', f'models:/{REGISTERED_MODEL_NAME}@candidate')

Reusing registered version 1
@candidate -> BankAssist-Intent-Classifier version 1
Source run: d8a6d09807e74267a643c038cb2ab659
Candidate URI: models:/BankAssist-Intent-Classifier@candidate


![Comparison of Model A and Model B in MLflow](../docs/images/candidate.png)

## 7. Validate `@candidate`

Load the registered candidate from MLflow, as the API will. Check that it is the selected logged model, reproduces the validation results, and returns a probability for each intent. Inspect a few validation predictions below. This step does not move `@champion`.

In [12]:
import numpy as np

candidate_uri = f'models:/{REGISTERED_MODEL_NAME}@candidate'
candidate_version = client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME, 'candidate'
)
candidate_model = mlflow.sklearn.load_model(candidate_uri)
candidate_predictions = candidate_model.predict(X_validation)
candidate_probabilities = candidate_model.predict_proba(X_validation)
candidate_macro_f1 = f1_score(
    y_validation, candidate_predictions, average='macro'
)
candidate_accuracy = accuracy_score(y_validation, candidate_predictions)

checks = {
    'selected logged model': candidate_version.source == selected_model_uri,
    'validation scores reproduced': (
        np.isclose(candidate_macro_f1, winner['validation_macro_f1'])
        and np.isclose(candidate_accuracy, winner['validation_accuracy'])
    ),
    'all intent classes present': list(candidate_model.classes_) == class_names,
    'one probability per intent': candidate_probabilities.shape == (
        len(X_validation), len(class_names)
    ),
    'probabilities sum to one': np.allclose(
        candidate_probabilities.sum(axis=1), 1.0
    ),
}
for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")
print(f'Candidate version: {candidate_version.version}')
print(f'Validation macro F1: {candidate_macro_f1:.4f}')
print(f'Validation accuracy: {candidate_accuracy:.4f}')
assert all(checks.values()), 'Candidate validation failed; do not promote.'

sample_predictions = pd.DataFrame({
    'message': X_validation.iloc[:5].to_numpy(),
    'actual_intent': y_validation.iloc[:5].to_numpy(),
    'predicted_intent': candidate_predictions[:5],
    'confidence': candidate_probabilities[:5].max(axis=1),
})
display(sample_predictions)

PASS: selected logged model
PASS: validation scores reproduced
PASS: all intent classes present
PASS: one probability per intent
PASS: probabilities sum to one
Candidate version: 1
Validation macro F1: 0.9194
Validation accuracy: 0.9189


,message,actual_intent,predicted_intent,confidence
0,the card payment didn't work,change_pin,card_payment_fee_charged,0.246220
1,is it acceptable to use either visa or masterc...,visa_or_mastercard,visa_or_mastercard,0.688172
2,can you explain why my payment was charged a fee?,card_payment_fee_charged,card_payment_fee_charged,0.768335
3,does it cost extra to send out more than one c...,getting_spare_card,getting_spare_card,0.400853
4,why was the price higher when i paid with my c...,card_payment_fee_charged,card_payment_fee_charged,0.730233


Validation checks two things: 
- that we can load the model through the registry’s @candidate alias
- that the loaded pipeline behaves as expected on validation messages. 

Section 7 checks its scores, intent classes, and probability output. I also produced a table showing five messages with their actual intent, predicted intent, and confidence so we can inspect concrete predictions.

Validation row 0 appears incorrectly labeled change_pin. The model predicts card_payment_fee_charged with low confidence (0.246), which also does not clearly match the message. This row will be flagged for human review.

## 8. Apply the promotion gate

Require the registered candidate to pass the checks above and reach at least **0.90 validation macro F1**. Report its result on the held-out test set once; do not use the test score to change the gate or select another model.

In [13]:
MIN_VALIDATION_MACRO_F1 = 0.90

X_test, y_test = test_df['text'], test_df['label']
test_predictions = candidate_model.predict(X_test)
test_macro_f1 = f1_score(y_test, test_predictions, average='macro')
test_accuracy = accuracy_score(y_test, test_predictions)

promotion_checks = {
    'candidate validation checks': all(checks.values()),
    'validation macro F1 >= 0.90': candidate_macro_f1 >= MIN_VALIDATION_MACRO_F1,
}
for check, passed in promotion_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")
print(f'Test macro F1 (final report): {test_macro_f1:.4f}')
print(f'Test accuracy (final report): {test_accuracy:.4f}')
promote = all(promotion_checks.values())
print('Promotion gate:', 'PASS' if promote else 'FAIL')

PASS: candidate validation checks
PASS: validation macro F1 >= 0.90
Test macro F1 (final report): 0.9674
Test accuracy (final report): 0.9664
Promotion gate: PASS


## 9. Promote to `@champion`

If the gate passes, assign `@champion` to the validated version. The alias identifies the approved model for the future API. Chetan can test the HTTP endpoint when the app exists.

In [14]:
assert promote, 'Promotion gate failed; do not move @champion.'
current_candidate = client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME, 'candidate'
)
assert current_candidate.version == candidate_version.version, (
    '@candidate changed since validation; validate it again.'
)

versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
previous_champion = next(
    (version for version in versions if 'champion' in version.aliases), None
)
print('Previous @champion version:', (
    previous_champion.version if previous_champion else 'none'
))

client.set_registered_model_alias(
    REGISTERED_MODEL_NAME, 'champion', current_candidate.version
)
champion = client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME, 'champion'
)
assert champion.version == current_candidate.version
print(f'@champion -> {REGISTERED_MODEL_NAME} version {champion.version}')
print(f'API model URI: models:/{REGISTERED_MODEL_NAME}@champion')

Previous @champion version: none
@champion -> BankAssist-Intent-Classifier version 1
API model URI: models:/BankAssist-Intent-Classifier@champion


![Comparison of Model A and Model B in MLflow](../docs/images/champion.png)

## What is next: API handoff

On this computer, the promotion gate passed and `@champion` points to version 1. MLflow stores the runs and model locally, outside Git, so Chetan will need to reproduce this registry on his own computer before his API can load the model.

- **Reproduce on Chetan's computer:** after the project files are committed and shared, set up the project environment, start `docker compose up -d mlflow`, and run this notebook from the top through section 9. The split CSVs are already in the repository; notebooks 1 and 2 do not need to be rerun. His MLflow run IDs and version number may differ.
- **Model URI:** `models:/BankAssist-Intent-Classifier@champion`. Load it with `mlflow.sklearn.load_model(...)` to use `predict` and `predict_proba`.
- **MLflow connection on his computer:** use `http://localhost:5001` from a local notebook or API process, or `http://mlflow:5000` from an API container in the same Compose project.
- **API contract:** implement `POST /predict` with the request and response fields in [the team plan](../docs/team-plan.md): `intent`, `confidence`, `route`, and `requires_human_review`. Agree with the team on the confidence threshold and routing rules.
- **End-to-end check:** send a sample request through the API and verify the response. If the API caches the model at startup, restart or reload it after a future `@champion` change.
- **Data-quality follow-up:** validation row 0 is flagged for human review above; do not silently change its label in the existing tracked experiment.